# 03 — Controlled Pose-only 29-frame model

This isolates the BODY25 pose representation under the same 29-frame temporal structure and grouped split used by the RGB and multimodal models.

The pose preprocessing follows Attempt 5: low-confidence/missing x/y coordinates are temporally interpolated, non-finite values are replaced with zero, and the confidence channel is retained.

In [ ]:
from pathlib import Path
import os, json, random
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, callbacks
from scipy.stats import spearmanr, pearsonr

REVISION_ROOT = Path(os.environ.get("AQA_REVISION_ROOT", str(Path.cwd() / "artifacts"))).expanduser().resolve()
CACHE_DIR = REVISION_ROOT / "cache"
RESULTS_DIR = REVISION_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(CACHE_DIR / "manifest_with_split.csv")
y_all = np.load(CACHE_DIR / "y_scores.npy")
train_idx = np.load(CACHE_DIR / "train_idx.npy")
val_idx = np.load(CACHE_DIR / "val_idx.npy")
test_idx = np.load(CACHE_DIR / "test_idx.npy")

assert len(manifest) == len(y_all)
print("Samples:", len(manifest))
print("Split:", len(train_idx), len(val_idx), len(test_idx))

PHASE_WEIGHTS = np.asarray([0.25, 0.50, 0.25], dtype=np.float32)
PHASE_NAMES = ["Buildup", "Execution", "FollowThrough"]
BODY_PARTS = ["Head", "Shoulders", "Hands", "Hips", "Feet"]

def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass

def safe_spearman(a, b):
    if np.unique(a).size < 2 or np.unique(b).size < 2:
        return np.nan
    return float(spearmanr(a, b).correlation)

def safe_pearson(a, b):
    if np.unique(a).size < 2 or np.unique(b).size < 2:
        return np.nan
    return float(pearsonr(a, b)[0])

def overall_scalar(y_phase, weights=PHASE_WEIGHTS):
    phase_scalar = np.asarray(y_phase).mean(axis=2)
    overall = np.sum(phase_scalar * np.asarray(weights)[None, :], axis=1)
    return phase_scalar, overall

def calculate_metrics(y_true, y_pred):
    true_phase, true_overall = overall_scalar(y_true)
    pred_phase, pred_overall = overall_scalar(y_pred)

    result = {
        "overall_SRC": safe_spearman(true_overall, pred_overall),
        "overall_Pearson": safe_pearson(true_overall, pred_overall),
        "overall_MAE": float(np.mean(np.abs(true_overall - pred_overall))),
        "overall_RMSE": float(np.sqrt(np.mean((true_overall - pred_overall)**2))),
    }

    for p, name in enumerate(PHASE_NAMES):
        result[f"{name}_SRC"] = safe_spearman(true_phase[:, p], pred_phase[:, p])
        result[f"{name}_Pearson"] = safe_pearson(true_phase[:, p], pred_phase[:, p])
        result[f"{name}_MAE"] = float(np.mean(np.abs(true_phase[:, p] - pred_phase[:, p])))

    return result

def common_head(z, name_prefix):
    # Common latent size/head across all ablation models.
    z = layers.Dense(
        128, activation="relu",
        kernel_regularizer=regularizers.l2(1e-4),
        name=f"{name_prefix}_project"
    )(z)
    z = layers.Dropout(0.25, name=f"{name_prefix}_project_dropout")(z)
    z = layers.Dense(
        96, activation="relu",
        kernel_regularizer=regularizers.l2(1e-4),
        name=f"{name_prefix}_dense96"
    )(z)
    z = layers.Dropout(0.35, name=f"{name_prefix}_drop96")(z)
    z = layers.Dense(
        48, activation="relu",
        kernel_regularizer=regularizers.l2(1e-4),
        name=f"{name_prefix}_dense48"
    )(z)
    z = layers.Dropout(0.25, name=f"{name_prefix}_drop48")(z)
    return layers.Dense(5, activation="linear", name=f"{name_prefix}_scores")(z)

def compile_model(model):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
        loss="mse",
        metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")],
    )
    return model

def make_callbacks(run_dir):
    return [
        callbacks.EarlyStopping(
            monitor="val_loss", patience=12, restore_best_weights=True, min_delta=1e-3
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1
        ),
        callbacks.ModelCheckpoint(
            filepath=str(run_dir / "best.weights.h5"),
            monitor="val_loss",
            save_best_only=True,
            save_weights_only=True,
            verbose=0,
        ),
    ]

def save_run(model_name, seed, model, history, y_train, pred_train, y_val, pred_val, y_test, pred_test):
    run_dir = RESULTS_DIR / model_name / f"seed_{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)

    metrics = {"model": model_name, "seed": int(seed)}
    for split_name, yt, yp in [
        ("train", y_train, pred_train),
        ("validation", y_val, pred_val),
        ("test", y_test, pred_test),
    ]:
        mm = calculate_metrics(yt, yp)
        metrics.update({f"{split_name}_{k}": v for k, v in mm.items()})

    best_epoch = int(np.argmin(history.history["val_loss"]) + 1)
    metrics["best_epoch"] = best_epoch

    pd.DataFrame(history.history).to_csv(run_dir / "history.csv", index=False)
    pd.DataFrame([metrics]).to_csv(run_dir / "metrics.csv", index=False)

    np.savez_compressed(
        run_dir / "predictions.npz",
        train_idx=train_idx, val_idx=val_idx, test_idx=test_idx,
        y_train=y_train, pred_train=pred_train,
        y_val=y_val, pred_val=pred_val,
        y_test=y_test, pred_test=pred_test,
    )

    with open(run_dir / "config.json", "w", encoding="utf-8") as f:
        json.dump(
            {
                "model": model_name,
                "seed": int(seed),
                "optimizer": "Adam",
                "learning_rate": 3e-4,
                "loss": "MSE",
                "epochs_max": 120,
                "batch_size": 32,
                "early_stopping_patience": 12,
                "reduce_lr_patience": 5,
                "phase_weights_used_for_reporting_only": [0.25, 0.50, 0.25],
                "note": "Model predicts the 3x5 phase/body-part matrix directly. Phase weights are not used in training loss."
            },
            f,
            indent=2,
        )
    return metrics

In [ ]:
pose29 = np.load(CACHE_DIR / "pose29_clean.npy", mmap_mode="r")
print("Pose tensor:", pose29.shape)

x_train = np.asarray(pose29[train_idx], dtype=np.float32)
x_val = np.asarray(pose29[val_idx], dtype=np.float32)
x_test = np.asarray(pose29[test_idx], dtype=np.float32)

PHASE_SLICES_29 = [(0, 14), (14, 21), (21, 29)]

def build_pose_transformer(input_shape=(29, 25, 3), embed_dim=64, num_heads=2, ff_dim=128, num_layers=1, drop=0.2):
    inp = layers.Input(shape=input_shape, name="pose_encoder_input")
    x = layers.Reshape((input_shape[0], input_shape[1] * input_shape[2]))(inp)
    x = layers.Dense(embed_dim, kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(drop)(x)

    for i in range(num_layers):
        attn = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim // num_heads, dropout=drop,
            name=f"pose_mha_{i}"
        )(x, x)
        x = layers.Add()([x, attn])
        x = layers.LayerNormalization()(x)

        ff = layers.Dense(ff_dim, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
        ff = layers.Dropout(drop)(ff)
        ff = layers.Dense(embed_dim, kernel_regularizer=regularizers.l2(1e-4))(ff)
        x = layers.Add()([x, ff])
        x = layers.LayerNormalization()(x)

    return models.Model(inp, x, name="PoseTransformerSmall")

def build_pose29_model():
    inp = layers.Input(shape=(29, 25, 3), name="pose29")
    noisy = layers.GaussianNoise(0.01, name="pose_noise")(inp)
    feats = build_pose_transformer()(noisy)

    outs = []
    for p, (s, e) in enumerate(PHASE_SLICES_29):
        pooled = layers.GlobalAveragePooling1D(name=f"phase{p}_pose_pool")(feats[:, s:e, :])
        outs.append(common_head(pooled, f"phase{p}"))

    return models.Model(inp, layers.Lambda(lambda z: tf.stack(z, axis=1), name="phase_scores")(outs))

model_check = build_pose29_model()
print("Output shape:", model_check.output_shape)
assert model_check.output_shape == (None, 3, 5)

In [ ]:
SEEDS = [42, 123, 2026]
all_run_metrics = []

y_train = y_all[train_idx]
y_val = y_all[val_idx]
y_test = y_all[test_idx]

for seed in SEEDS:
    print("\n" + "="*80)
    print("MODEL:", "POSE29_GROUPED", "| SEED:", seed)
    print("="*80)

    tf.keras.backend.clear_session()
    set_seed(seed)

    run_dir = RESULTS_DIR / "POSE29_GROUPED" / f"seed_{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)

    model = build_pose29_model()
    model = compile_model(model)
    model.summary()

    history = model.fit(
        x_train,
        y_train,
        validation_data=(x_val, y_val),
        epochs=120,
        batch_size=32,
        callbacks=make_callbacks(run_dir),
        verbose=1,
        shuffle=True,
    )

    pred_train = model.predict(x_train, batch_size=64, verbose=0)
    pred_val = model.predict(x_val, batch_size=64, verbose=0)
    pred_test = model.predict(x_test, batch_size=64, verbose=0)

    metrics = save_run(
        "POSE29_GROUPED", seed, model, history,
        y_train, pred_train, y_val, pred_val, y_test, pred_test
    )
    all_run_metrics.append(metrics)

summary_df = pd.DataFrame(all_run_metrics)
display(summary_df)

summary_path = RESULTS_DIR / "POSE29_GROUPED" / "all_seeds_metrics.csv"
summary_df.to_csv(summary_path, index=False)
print("\nSaved:", summary_path)